# Multi-Cancer Image Classification Using a Hybrid CNN–Mamba Architecture with CBAM, DropBlock, and Grad-CAM++


## 1. Research Overview and Experimental Configuration


In [ ]:
# Integrated Mamba, dropblock and trained on 120k images having 24 classes
# Trained upto 10 epochs
# 70:20:10 split
# 

## 2. Library Imports and Environment Setup


In [1]:
import os
import json
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from torchvision import transforms, models
from torchvision.ops import DropBlock2d
import time

In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install causal-conv1d==1.4.0 && pip install mamba-ssm==2.2.2

In [ ]:
from mamba_ssm import Mamba

## 3. Dataset Acquisition and Loading


In [ ]:
import kagglehub
path = kagglehub.dataset_download("obulisainaren/multi-cancer")
root_dir = os.path.join(path, "Multi Cancer", "Multi Cancer")

# Analyze
#results = analyze_multi_cancer_dataset(root_dir)

In [ ]:
class MultiCancerDataset(Dataset):
    def __init__(self, root_dir, transform=None, exclude_types=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_names = []
        self.label_map = {}
        self.exclude_types = exclude_types or []

        cancer_types = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and d.lower() not in self.exclude_types
        ])

        label_counter = 0
        for cancer_type in cancer_types:
            cancer_path = os.path.join(root_dir, cancer_type)
            class_folders = sorted([
                d for d in os.listdir(cancer_path)
                if os.path.isdir(os.path.join(cancer_path, d))
            ])

            for class_folder in class_folders:
                class_path = os.path.join(cancer_path, class_folder)
                image_files = [
                    f for f in os.listdir(class_path)
                    if f.lower().endswith(('.jpg', '.png', '.jpeg', '.bmp', '.tif'))
                ]

                label_name = f"{cancer_type}_{class_folder}"
                self.label_map[label_name] = label_counter

                for img_name in image_files:
                    self.images.append(os.path.join(class_path, img_name))
                    self.labels.append(label_counter)

                self.class_names.append(label_name)
                label_counter += 1

        print(f" Total images loaded: {len(self.images)}")
        print(f" Total classes used: {len(self.class_names)}")
        print(f" Excluded types: {self.exclude_types}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

## 4. Image Preprocessing and Data Augmentation


In [ ]:
train_transform = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.RandomRotation(degrees=15),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
class TransformDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

## 5. Dataset Preparation and Train–Validation–Test Splitting


In [ ]:
full_dataset = MultiCancerDataset(root_dir, transform=None, exclude_types=["oral cancer"])

In [ ]:
total_size = len(full_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.20 * total_size)
test_size = total_size - train_size - val_size

In [ ]:
train_dataset, val_dataset, test_dataset = random_split( full_dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42) )

train_dataset = TransformDataset(train_dataset, train_transform)
val_dataset = TransformDataset(val_dataset, val_transform)
test_dataset = TransformDataset(test_dataset, val_transform)

In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
print(f"Training samples: {len(train_dataset)} ({train_size/total_size*100:.1f}%)")
print(f"Validation samples: {len(val_dataset)} ({val_size/total_size*100:.1f}%)")
print(f"Test samples: {len(test_dataset)} ({test_size/total_size*100:.1f}%)")

## 6. Dataset Sample Visualization (Optional)


In [ ]:
# def show_sample_images(dataset, num_samples=8):
#     # Create a simple transform for display (without normalization)
#     display_transform = transforms.Compose([
#         transforms.Resize((224, 224)),
#         transforms.ToTensor()
#     ])

#     # Create a temporary dataset for display
#     temp_dataset = BreastCancerDataset(root_dir, transform=display_transform)

#     # Get random indices
#     indices = np.random.choice(len(temp_dataset), num_samples, replace=False)

#     # Set up the plot
#     fig, axes = plt.subplots(2, 2, figsize=(8, 8))
#     axes = axes.ravel()

#     class_names = ['Benign', 'Malignant']

#     for i, idx in enumerate(indices):
#         image, label = temp_dataset[idx]

#         # Convert tensor to numpy and transpose for matplotlib
#         if isinstance(image, torch.Tensor):
#             image_np = image.permute(1, 2, 0).numpy()
#         else:
#             image_np = np.array(image)

#         # Display the image
#         axes[i].imshow(image_np)
#         axes[i].set_title(f'Class: {class_names[label]}', fontsize=12, fontweight='bold')
#         axes[i].axis('off')
#     plt.tight_layout()
#     plt.show()
# show_sample_images(full_dataset, num_samples=4)

## 7. Convolutional Block Attention Module (CBAM)


In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

## 8. Mamba-Based Feature Sequence Modeling


In [ ]:
class FeatureToSequence(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # x shape: (batch, channels, height, width)
        batch, channels, h, w = x.shape
        # Reshape to (batch, height*width, channels)
        x = x.flatten(2).permute(0, 2, 1)
        return x, (h, w)

class SequenceToFeature(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, spatial_dims):
        # x shape: (batch, seq_len, channels)
        batch, seq_len, channels = x.shape
        h, w = spatial_dims
        # Reshape back to (batch, channels, height, width)
        x = x.permute(0, 2, 1).reshape(batch, channels, h, w)
        return x

class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.to_seq = FeatureToSequence()
        self.mamba = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand
        )
        self.to_feat = SequenceToFeature()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # Convert to sequence
        x_seq, spatial_dims = self.to_seq(x)

        # Apply Mamba
        x_seq = self.norm(x_seq)
        x_seq = self.mamba(x_seq)

        # Convert back to feature map
        x = self.to_feat(x_seq, spatial_dims)
        return x

## 9. Proposed CNN–Mamba Hybrid Architecture


In [ ]:
class CNNMambaHybrid(nn.Module):
    def __init__(self, num_classes=24):
        super(CNNMambaHybrid, self).__init__()

        # Block 1 - No attention
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2 - Added CBAM
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.cbam2 = CBAM(64)
        self.dropblock2 = DropBlock2d(p=0.1, block_size=5)
        self.pool2 = nn.MaxPool2d(2, 2)

        # First Mamba (mid-level spatial relationships)
        self.mamba1 = MambaBlock(d_model=64)

        # Block 3 - No attention
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Block 4 - Added CBAM
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU(inplace=True)
        self.cbam4 = CBAM(256)
        self.dropblock4 = DropBlock2d(p=0.2, block_size=7)
        self.pool4 = nn.MaxPool2d(2, 2)

        # Second Mamba (high-level spatial relationships)
        self.mamba2 = MambaBlock(d_model=256)

        # Classification head
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.output = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.dropblock2(self.cbam2(self.relu2(self.bn2(self.conv2(x))))))

        # First Mamba
        x = self.mamba1(x)

        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.pool4(self.dropblock4(self.cbam4(self.relu4(self.bn4(self.conv4(x))))))

        # Second Mamba
        x = self.mamba2(x)

        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.output(x)
        return x

## 10. Model Initialization and Training Configuration


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
model = CNNMambaHybrid(num_classes=24).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))

## 11. Model Training, Early Stopping, and Checkpoint Management


In [ ]:
def resume_training_fresh(
    model, train_loader, val_loader, criterion, optimizer,
    additional_epochs=5,
    checkpoint_path='/kaggle/working/checkpoint_v2.pth',
    best_model_path='/kaggle/working/best_model_v2.pth',
    history_file='/kaggle/working/training_history_v2.json',
    patience=3,
    device='cuda'
):
    if os.path.exists(checkpoint_path):
        print("Checkpoint found. Resuming training...")

        checkpoint = torch.load(checkpoint_path, map_location=device)

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        start_epoch = checkpoint['epoch']
        train_losses = checkpoint['train_losses']
        val_losses = checkpoint['val_losses']
        train_accuracies = checkpoint['train_accuracies']
        val_accuracies = checkpoint['val_accuracies']
        best_val_acc = checkpoint['best_val_acc']
        best_epoch = checkpoint.get('best_epoch', start_epoch)

        print(f"Resuming from epoch {start_epoch}")
        print(f"Previous best val acc: {best_val_acc:.2f}%")
    else:
        print("No checkpoint found. Starting fresh training...")
        start_epoch = 0
        train_losses, val_losses = [], []
        train_accuracies, val_accuracies = [], []
        best_val_acc = 0.0
        best_epoch = 0
        
    no_improve = 0
    start_time = time.time()
    for epoch in range(start_epoch, start_epoch + additional_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{start_epoch+additional_epochs} - Training')
        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{start_epoch+additional_epochs} - Validation')
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Acc': f'{100.*val_correct/val_total:.2f}%'
                })
        
        # Calculate epoch metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_val_loss = val_loss / len(val_loader)
        epoch_train_acc = 100. * train_correct / train_total
        epoch_val_acc = 100. * val_correct / val_total
        
        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        train_accuracies.append(epoch_train_acc)
        val_accuracies.append(epoch_val_acc)
        
        print(f'Epoch [{epoch+1}/{start_epoch+additional_epochs}]:')
        print(f'  Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
        print(f'  Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')
        
        # Save best model
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), best_model_path)
            # shutil.copy(best_model_path, "/content/drive/MyDrive/kaggle/best_model_v2.pth")
            print(f"Best model updated! Val Acc = {best_val_acc:.2f}%")
            no_improve = 0  
        else:
            no_improve += 1
            print(f"No improvement for {no_improve}/{patience} epochs.")

            if no_improve >= patience:
                print("Early stopping triggered!")
                break

        # Save checkpoint
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies,
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch
        }, checkpoint_path)   
        # shutil.copy(checkpoint_path, "/content/drive/MyDrive/kaggle/checkpoint_v2.pth")
        print("Checkpoint saved: checkpoint.pth")
        print('-' * 60)
        
        # Save history
        with open(history_file, 'w') as f:
            json.dump({
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_accuracies': train_accuracies,
                'val_accuracies': val_accuracies,
                'best_val_acc': best_val_acc,
                'best_epoch': best_epoch
            }, f)
        # shutil.copy(history_file, "/content/drive/MyDrive/kaggle/training_history_v2.json")
        # print("History backed up to Google Drive.")
    
    end_time = time.time()
    total_time = end_time - start_time
    minutes, seconds = divmod(total_time, 60)
    print(f"\nTotal Training Time: {int(minutes)} min {int(seconds)} sec")
    print(f"Best Model - Epoch: {best_epoch}, Val Acc: {best_val_acc:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
# train_losses, val_losses, train_accuracies, val_accuracies = resume_training_fresh(
#     model, train_loader, val_loader, criterion, optimizer, 
#     additional_epochs=1, 
#     checkpoint_path='/kaggle/working/checkpoint_v2.pth',
#     best_model_path='/kaggle/working/best_model_v2.pth',
#     history_file='/kaggle/working/training_history_v2.json',
#     patience=5,
#     device=device
# )

## 12. Best Model Checkpoint Loading


In [ ]:
# Load best checkpoint
best_model_path = '/kaggle/input/models/ashfaqbinhoque/multi-cancer-model-new/pytorch/default/1/best_model_v2.pth'
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print("Loaded best model weights.")

## 13. Test Set Evaluation and Classification Performance Metrics


In [ ]:
def evaluate_model(model, test_loader, device, dataset_name="Test"):
    model.eval()
    test_correct = 0
    test_total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc=f'Evaluating on {dataset_name} Set')
        for images, labels in test_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            test_pbar.set_postfix({
                'Acc': f'{100.*test_correct/test_total:.2f}%'
            })
    
    test_accuracy = 100 * test_correct / test_total
    return test_accuracy, all_predictions, all_labels

In [ ]:
# Evaluate on test set
test_accuracy, test_predictions, test_labels = evaluate_model(model, test_loader, device)

print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Total Test Samples: {len(test_labels)}")
print(f"Correct Predictions: {sum(np.array(test_predictions) == np.array(test_labels))}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"1. Accuracy: {accuracy_score(test_labels, test_predictions):.4f}")
print(f"2. Precision (Weighted): {precision_score(test_labels, test_predictions, average='weighted'):.4f}")
print(f"3. Recall (Weighted): {recall_score(test_labels, test_predictions, average='weighted'):.4f}")
print(f"4. F1-Score (Weighted): {f1_score(test_labels, test_predictions, average='weighted'):.4f}")

## 14. Training History and Learning Curve Analysis


In [ ]:
# Load training history
history_file = '/kaggle/input/datasets/ashfaqbinhoque/training-history-json'
with open(history_file, 'r') as f:
    history = json.load(f)

# Extract metrics
train_losses = history['train_losses']
val_losses = history['val_losses']
train_accuracies = history['train_accuracies']
val_accuracies = history['val_accuracies']
best_epoch = history['best_epoch']
best_val_acc = history['best_val_acc']

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
ax1.plot(train_losses, label='Training Loss', linewidth=2, marker='o', markersize=5)
ax1.plot(val_losses, label='Validation Loss', linewidth=2, marker='s', markersize=5)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot accuracies
ax2.plot(train_accuracies, label='Training Accuracy', linewidth=2, marker='o', markersize=5)
ax2.plot(val_accuracies, label='Validation Accuracy', linewidth=2, marker='s', markersize=5)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 15. Confusion Matrix Analysis


In [ ]:
cm = confusion_matrix(test_labels, test_predictions)

plt.figure(figsize=(20, 18))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, square=True, cbar_kws={"shrink": 0.8})
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 16. Multiclass ROC Curve and AUC Analysis


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, roc_curve, auc

all_probs = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Convert to numpy arrays
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

num_classes = len(full_dataset.class_names)  # should be 24
y_bin = label_binarize(all_labels, classes=list(range(num_classes)))

roc_auc = roc_auc_score(y_bin, all_probs, average='weighted', multi_class='ovr')
print(f"ROC-AUC Score (Weighted, OvR): {roc_auc:.4f}")

plt.figure(figsize=(15, 12))

for class_id in range(num_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, class_id], all_probs[:, class_id])
    roc_auc_cls = auc(fpr, tpr)

    class_name = full_dataset.class_names[class_id]
    plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc_cls:.3f})', linewidth=1.3)

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves for All 24 Cancer Classes', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 17. Model Interpretability Using Grad-CAM++


In [ ]:

# ====================== Grad-CAM++ for CNNMambaHybrid (20 Images) ======================

import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random

# ---------- 1. Choose a target layer ----------

activations = None
gradients = None

def forward_hook(module, input, output):
    global activations
    activations = output

def backward_hook(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

# Register hooks on the second Mamba block
handle_f = model.mamba2.register_forward_hook(forward_hook)
handle_b = model.mamba2.register_full_backward_hook(backward_hook)

# ---------- 2. Select 20 random test images ----------

num_images = 20
random_indices = random.sample(
    range(len(test_dataset)),
    min(num_images, len(test_dataset))
)

model.eval()

# Image normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Create a figure with 20 rows and 3 columns
fig, axes = plt.subplots(
    len(random_indices), 3,
    figsize=(14, 5 * len(random_indices))
)

# ---------- 3. Process each test image ----------

for i, idx in enumerate(random_indices):

    img_tensor, true_label = test_dataset[idx]
    true_class_name = full_dataset.class_names[true_label]

    # Denormalize image for visualization
    img_denorm = img_tensor * std + mean
    img_denorm = torch.clamp(img_denorm, 0, 1)
    img_np = img_denorm.permute(1, 2, 0).numpy()

    input_tensor = img_tensor.unsqueeze(0).to(device)

    # ---------- 4. Forward + Backward ----------

    model.zero_grad()

    output = model(input_tensor)

    pred_idx = output.argmax(dim=1).item()
    pred_class_name = full_dataset.class_names[pred_idx]

    score = output[0, pred_idx]
    confidence = torch.softmax(output, dim=1)[0, pred_idx].item()

    score.backward()

    # ---------- 5. Grad-CAM++ calculation ----------

    grads = gradients[0]       # (C, H, W)
    acts = activations[0]      # (C, H, W)

    grads_pos = F.relu(grads)
    grads_2 = grads ** 2
    grads_3 = grads_2 * grads

    sum_acts = torch.sum(acts, dim=(1, 2), keepdim=True)

    alpha_num = grads_2
    alpha_denom = 2 * grads_2 + sum_acts * grads_3

    alpha_denom = torch.where(
        alpha_denom != 0.0,
        alpha_denom,
        torch.ones_like(alpha_denom)
    )

    alphas = alpha_num / alpha_denom

    weights = torch.sum(
        alphas * grads_pos,
        dim=(1, 2)
    )

    cam = torch.sum(
        weights.view(-1, 1, 1) * acts,
        dim=0
    )

    cam = F.relu(cam)

    # Normalize CAM
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    # ---------- 6. Resize CAM ----------

    cam_np = cam.detach().cpu().numpy()

    cam_resized = np.array(
        Image.fromarray(
            np.uint8(255 * cam_np)
        ).resize((224, 224), Image.BILINEAR)
    ) / 255.0

    # ---------- 7. Visualize ----------

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(
        f"Original\nTrue: {true_class_name}",
        fontsize=10
    )
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam_resized, cmap="jet")
    axes[i, 1].set_title(
        "Grad-CAM++ Heatmap",
        fontsize=10
    )
    axes[i, 1].axis("off")

    axes[i, 2].imshow(img_np)
    axes[i, 2].imshow(cam_resized, cmap="jet", alpha=0.45)
    axes[i, 2].set_title(
        f"Overlay\nPred: {pred_class_name}\nConfidence: {confidence:.4f}",
        fontsize=10
    )
    axes[i, 2].axis("off")

    print(
        f"Image {i+1}/20 | True: {true_class_name} | "
        f"Predicted: {pred_class_name} | Confidence: {confidence:.4f}"
    )

# ---------- 8. Final figure ----------

plt.suptitle(
    "Explainable AI – Grad-CAM++ on CNNMambaHybrid (20 Test Images)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# ---------- 9. Clean up hooks ----------

handle_f.remove()
handle_b.remove()

print("Grad-CAM++ completed for all test images!")
